In [ ]:
# hide
# no-output
import pathlib
import numpy as np
import pyquist as pq

# Load Freesound sounds from the copies cached with this book (no API key needed)
pq.paths.CACHE_DIR = pathlib.Path("../assets")
from pyquist.web import fetch_freesound

In [ ]:
def freesound(fsid: int) -> pq.Audio:
    """An instrument that plays the Freesound sound with ID `fsid`."""
    audio, _ = fetch_freesound(fsid)
    audio = audio.as_mono()
    if audio.sample_rate != 44100:
        audio = audio.resample(44100)         # all sounds in a score need one sample rate
    return audio


def ascii_to_score(ascii: str,
                   row_fsids: tuple[int, ...] = (332372, 46709, 673512)) -> pq.Score:
    """One row per drum, one column per step: an X plays that row's sound at that step."""
    rows = ascii.strip().splitlines()
    score = pq.Score()
    for step in range(len(rows[0])):
        for row, fsid in zip(rows, row_fsids):
            if row[step] == "X":
                score.append((step, {"fsid": fsid}))
    return score

In [ ]:
pattern = """
X-X-X-X-X-X-X-X-X-X-X-X-X-X-X-X-
----X-------X-------X-------X---
X-------X-------X-X-----X-------
"""                                           # rows: hi-hat, snare, kick

score = ascii_to_score(pattern)
print(score[:4])

# Each step is a sixteenth note: four steps per beat at 120 BPM
metronome = pq.BasicMetronome(bpm=120 * 4)
pq.play(score.render(freesound, metronome), normalize=True)